# Module 07 — Lab: MCP server in 30 lines

We use the server defined in `./server.py`. This notebook:
1. Verifies your `mcp` install.
2. Spawns the server as a stdio child process and talks MCP to it from Python.
3. Lists tools / resources / prompts and exercises each.

## 1. Install

In [ ]:
# Run once in your venv:
# !pip install mcp
import importlib
assert importlib.util.find_spec('mcp'), 'pip install mcp first'
print('mcp ok')

## 2. Spawn the server and talk to it

In [ ]:
import sys, asyncio, json, pathlib
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp.client.session import ClientSession

SERVER = str(pathlib.Path('server.py').resolve())

async def main():
    params = StdioServerParameters(command=sys.executable, args=[SERVER])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print('TOOLS:', [t.name for t in tools.tools])

            resources = await session.list_resource_templates()
            print('RESOURCES:', [r.uriTemplate for r in resources.resourceTemplates])

            prompts = await session.list_prompts()
            print('PROMPTS:', [p.name for p in prompts.prompts])

            # Call the add tool
            r = await session.call_tool('add', {'a': 2, 'b': 40})
            print('add(2,40) ->', r.content[0].text)

            # Read a resource
            note = await session.read_resource('notes://rag')
            print('notes://rag ->', note.contents[0].text)

            # Render a prompt
            p = await session.get_prompt('code_review', {'language': 'Python'})
            print('prompt code_review(Python) ->', p.messages[0].content.text[:120], '...')

asyncio.run(main())

## 3. Wire it into Claude Code

Add to your Claude Code settings (`~/.claude.json` on macOS/Linux, `%USERPROFILE%\.claude.json` on Windows):

```json
{
  "mcpServers": {
    "course-demo": {
      "command": "python",
      "args": ["D:/Lourdu-Personal/claude/07-mcp/server.py"]
    }
  }
}
```

Then restart Claude Code and try:
- Running the slash command `/course-demo:code_review`.
- Asking Claude in a session: "Use add to compute 17+25."

You should see Claude Code call your MCP tool live.